# 01 — Facial Image Preprocessing

## Overview

This notebook prepares the facial images used as the baseline condition for the image-compression experiment.

The preprocessing stage validates face position and image quality before isolating and standardizing the facial region. Accepted images are resized to **112 × 112 pixels** and saved in **PNG format** to preserve a lossless baseline prior to JPEG, JPEG XL, and HEIC compression.

### Pipeline

Raw facial images  
→ face and landmark detection  
→ head-pose validation  
→ blur and brightness validation  
→ facial masking and cropping  
→ 112 × 112 normalization  
→ lossless PNG baseline images

> **Privacy:** The original participant facial images are research data and are not distributed with the public repository.

## Environment Setup

The original experiment was executed in Google Colab and used version-specific package installation to maintain compatibility with MediaPipe and its dependencies.

This setup cell is preserved from the working experiment. It changes the active Python environment, so it should only be run in an isolated environment such as a fresh Colab runtime.

In [ ]:
# --- VERSION-STABILIZED COLAB SETUP ---
print("⚙️ Force-cleaning environment packages...")

# Remove conflicting dependencies to allow clean reconstruction
!pip uninstall -y mediapipe protobuf tensorflow google-api-core proto-plus

print("📦 Installing modern, patched production versions...")
# Install matching protobuf and structural requirements first
!pip install -q protobuf==4.25.3 google-api-core proto-plus
# Install the stable verified media processing ecosystem
!pip install -q mediapipe==0.10.14
!pip install -q opencv-python numpy

print("📦 Installing Pillow-heif for HEIC support...")
!pip install -q pillow-heif

print("✔️ Pillow-heif installed.")

## Library Imports

The preprocessing workflow uses:

- `pathlib` and `os` for file and directory handling
- `shutil` for resetting output folders and copying rejected images
- `math` for pose-angle calculations
- OpenCV for image loading, masking, cropping, resizing, and quality checks
- MediaPipe Face Mesh for facial landmark detection
- NumPy for numerical and image-array operations

In [ ]:
from pathlib import Path
import math
import os
import shutil

import cv2
import mediapipe as mp
import numpy as np

import tempfile

from PIL import Image
from pillow_heif import register_heif_opener

## Dataset and Output Paths

The participant images are stored outside the public repository.

The notebook expects three private data directories:

- `raw_images/` — original participant images
- `preprocessed_images/` — accepted standardized baseline images
- `rejected_images/` — images rejected by the validation pipeline

The repository-level `data/` directory should remain excluded from Git.

In [ ]:
# -----------------------------
# Paths
# -----------------------------
INPUT_DIR = Path("../data/raw_images")
OUTPUT_DIR = Path("../data/preprocessed_images")
REJECTED_DIR = Path("../data/rejected_images")

## Preprocessing Configuration and Face Detection

The following values are the thresholds used in the original experiment and are preserved here without methodological changes.

### Head-pose criteria

- Maximum roll angle: **30°**
- Yaw-ratio threshold: **0.05**
- Maximum pitch ratio: **0.70**

### Image-quality criteria

- Minimum Laplacian variance for blur screening: **25.0**
- Minimum mean grayscale brightness: **15**

### Standardized output

Accepted face crops are resized to **112 × 112 pixels**.

MediaPipe Face Mesh is configured to detect a maximum of one face per image.

In [ ]:
# Preprocessing thresholds used in the original experiment
MAX_ROLL_DEGREES = 30.0
MAX_YAW_RATIO = 0.05
MAX_PITCH_RATIO = 0.70

MIN_BLUR_VARIANCE = 25.0
MIN_BRIGHTNESS = 15

OUTPUT_SIZE = (112, 112)

# Reset the preprocessing output directories before a full batch run.
print("Resetting preprocessing output folders...")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

if REJECTED_DIR.exists():
    shutil.rmtree(REJECTED_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)

# Initialize MediaPipe Face Mesh.
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.3,
)

print("Face-detection environment initialized.")

## Image Validation and Facial Normalization

This section defines the core preprocessing functions.

### Blur validation

Blur is measured using the variance of the Laplacian. Images with a value below the experimental threshold are rejected.

In [ ]:
def check_blur(face_crop):
    """Check whether the cropped face satisfies the blur threshold."""
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()

    return (variance >= MIN_BLUR_VARIANCE), variance

### Brightness validation

Brightness is measured as the mean grayscale pixel intensity. Images below the experimental threshold are rejected.

In [ ]:
def check_brightness(face_crop):
    """Check whether the cropped face satisfies the brightness threshold."""
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    mean_val = np.mean(gray)

    return (mean_val >= MIN_BRIGHTNESS), mean_val

### Head-pose validation

MediaPipe landmarks are used to calculate roll, yaw, and pitch indicators. The existing formulas and thresholds are preserved from the original experiment.

### Facial masking, cropping, and resizing

After validation:

1. A facial silhouette polygon is created from selected MediaPipe landmarks.
2. Pixels outside the facial region are masked.
3. The facial region is cropped to its bounding rectangle.
4. Blur and brightness checks are applied to the cropped face.
5. Accepted face crops are resized to **112 × 112 pixels** using `INTER_AREA`.

The resulting image is returned as the standardized baseline image.

In [ ]:
def check_pose(landmarks, img_w, img_h):
    """Calculates head alignment metrics across pitch, roll, and yaw planes."""
    # Index locations based on MediaPipe structural mapping blueprint
    nose_tip = landmarks[1]
    left_eye = landmarks[33]
    right_eye = landmarks[263]
    left_ear = landmarks[234]
    right_ear = landmarks[454]

    # 1. Roll calculation (Midline tilt angle calculation)
    dx = right_eye.x - left_eye.x
    dy = right_eye.y - left_eye.y
    roll_angle = abs(math.degrees(math.atan2(dy, dx)))

    # 2. Yaw calculation (Horizontal profile checking ratio analysis)
    dist_left = abs(nose_tip.x - left_ear.x)
    dist_right = abs(right_ear.x - nose_tip.x)
    total_yaw_span = dist_left + dist_right
    yaw_ratio = min(dist_left, dist_right) / total_yaw_span if total_yaw_span > 0 else 0

    # 3. Pitch calculation (Vertical inclination ratio map)
    top_lip = landmarks[0]
    mid_eyebrow = landmarks[168]
    dist_upper = abs(nose_tip.y - mid_eyebrow.y)
    dist_lower = abs(top_lip.y - nose_tip.y)
    total_v_span = dist_upper + dist_lower
    pitch_ratio = abs(dist_upper - dist_lower) / total_v_span if total_v_span > 0 else 0

    # Dynamic logic evaluation matching the relaxed metrics
    is_roll_ok = roll_angle <= MAX_ROLL
    is_yaw_ok = yaw_ratio >= MAX_YAW
    is_pitch_ok = pitch_ratio <= MAX_PITCH

    is_pose_valid = is_roll_ok and is_yaw_ok and is_pitch_ok
    status_msg = f"Roll:{roll_angle:.1f}°, YawRatio:{yaw_ratio:.2f}, PitchRatio:{pitch_ratio:.2f}"

    return is_pose_valid, status_msg

def process_pipeline(file_path, current_rej_dir):
    """Runs data standardization workflow on raw images using MediaPipe."""
    img = cv2.imread(file_path)
    if img is None:
        return None, "CORRUPT_OR_UNREADABLE"

    h, w, _ = img.shape
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_img)

    if not results.multi_face_landmarks:
        # Save a duplicate to reject repository if facial tissue recognition fails completely
        shutil.copy(file_path, os.path.join(current_rej_dir, f"NO_FACE_{os.path.basename(file_path)}"))
        return None, "NO_FACE_DETECTED"

    landmarks = results.multi_face_landmarks[0].landmark

    # Pose Gate Assessment
    pose_pass, pose_details = check_pose(landmarks, w, h)
    if not pose_pass:
        shutil.copy(file_path, os.path.join(current_rej_dir, f"FAIL_POSE_{os.path.basename(file_path)}"))
        return None, f"POSE_REJECTED ({pose_details})"

    # Silhouette Segmentation and Oval Isolation Logic
    # 36 specialized target coordinate indicators tracking the silhouette perimeter context
    silhouette_indices = [
        10,  338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
        397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
        172, 58,  132, 93,  234, 127, 162, 21,  54,  103, 67,  109
    ]

    polygon_points = np.array([
        [int(landmarks[idx].x * w), int(landmarks[idx].y * h)] for idx in silhouette_indices
    ], dtype=np.int32)

    # Create the dynamic isolation mask layer
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [polygon_points], 255)

    # Black out out-of-boundary variables (hair, clothes, backgrounds)
    masked_face = cv2.bitwise_and(img, img, mask=mask)

    # Crop to the targeted face bounding box
    x, y, crop_w, crop_h = cv2.boundingRect(polygon_points)
    # Clamp bounds to ensure safe array selection operations
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(w, x + crop_w), min(h, y + crop_h)
    face_crop = masked_face[y1:y2, x1:x2]

    if face_crop.size == 0:
        return None, "EMPTY_CROP_ERROR"

    # Technical Biometric Quality Screening (FIQA Gate Checks)
    sharp_pass, sharp_val = check_blur(face_crop)
    bright_pass, bright_val = check_brightness(face_crop)

    if not sharp_pass:
        shutil.copy(file_path, os.path.join(current_rej_dir, f"BLUR_v{sharp_val:.1f}_{os.path.basename(file_path)}"))
        return None, f"QUALITY_BLUR_REJECTED (Value: {sharp_val:.1f})"

    if not bright_pass:
        shutil.copy(file_path, os.path.join(current_rej_dir, f"DARK_v{bright_val:.1f}_{os.path.basename(file_path)}"))
        return None, f"QUALITY_DARK_REJECTED (Value: {bright_val:.1f})"

    # Final resolution downscaling grid adjustment matching ArcFace input bounds
    standardized_baseline = cv2.resize(face_crop, (112, 112), interpolation=cv2.INTER_AREA)
    return standardized_baseline, "SUCCESS"

## HEIC Input Handling

OpenCV does not directly load the HEIC files used in the original dataset workflow.

For HEIC inputs, this helper temporarily converts the image to JPEG so that the existing OpenCV preprocessing pipeline can read it. The temporary file is deleted after the image has been processed.

This conversion behavior is preserved from the original implementation.

In [ ]:
# Register the HEIF opener to Pillow
register_heif_opener()

def handle_image_format_conversion(original_file_path):
    """
    Checks if the image is in HEIC format and converts it to JPEG if necessary.
    Returns (processed_file_path, is_temp_file) where is_temp_file is True if a temporary
    file was created and needs to be deleted.
    """
    if original_file_path.lower().endswith('.heic'):
        try:
            img = Image.open(original_file_path)
            # Create a temporary file to save the converted image
            base_name = os.path.basename(original_file_path)
            temp_dir = tempfile.gettempdir() # Get system's temporary directory
            temp_jpg_name = os.path.splitext(base_name)[0] + '.jpg'
            temp_jpg_path = os.path.join(temp_dir, temp_jpg_name)

            img.save(temp_jpg_path, 'jpeg')
            return temp_jpg_path, True
        except Exception as e:
            print(f"Error converting HEIC file {original_file_path}: {e}")
            # If conversion fails, return original path but mark as not temp
            return original_file_path, False
    return original_file_path, False # Not a HEIC file, no temp file created

## Batch Preprocessing

The private dataset is organized into three demographic groups:

- Malay
- Chinese
- Indian

Each accepted image is saved to its corresponding output folder using a demographic code and a zero-padded sequential identifier.

For example:

```text
01_0001.png
02_0001.png
03_0001.png
```

The batch loop:

1. Locates supported image files.
2. Converts HEIC files to a temporary readable format when required.
3. Runs the validation and preprocessing pipeline.
4. Saves accepted images as PNG files.
5. Records rejected images in the private rejection directory.
6. Deletes temporary converted files.
7. Prints an aggregate processing summary.

> **Privacy reminder:** Rejection messages may print original filenames. Clear notebook outputs before committing a run based on private participant data.

In [ ]:
# Demographic groups and output filename prefixes
ethnicities = ["malay", "chinese", "indian"]

eth_codes = {
    "malay": "01",
    "chinese": "02",
    "indian": "03",
}

audit_summary = {}

print("=" * 58)
print("INITIALIZING FACIAL IMAGE PREPROCESSING")
print("=" * 58)

for eth in ethnicities:
    src_folder = os.path.join(INPUT_DIR, eth)
    tgt_folder = os.path.join(OUTPUT_DIR, eth)
    rej_folder = os.path.join(REJECTED_DIR, eth)

    os.makedirs(tgt_folder, exist_ok=True)
    os.makedirs(rej_folder, exist_ok=True)

    if not os.path.exists(src_folder):
        print(
            f"Source folder missing for '{eth}'. "
            "Skipping this demographic group."
        )

        audit_summary[eth] = {
            "Baseline": 0,
            "Rejected": 0,
        }
        continue

    # Supported image formats used in the experiment.
    all_files = os.listdir(src_folder)

    valid_images = sorted(
        [
            filename
            for filename in all_files
            if filename.lower().endswith(
                (".png", ".jpg", ".jpeg", ".webp", ".heic")
            )
        ]
    )

    print(
        f"\nProcessing [{eth.upper()}] "
        f"| Raw images found: {len(valid_images)}"
    )

    accepted_count = 0
    rejected_count = 0
    serial_counter = 1

    code_prefix = eth_codes[eth]

    for filename in valid_images:
        original_img_path = os.path.join(src_folder, filename)

        # Convert HEIC input to a temporary JPEG when required.
        processed_img_path, is_temp_file = (
            handle_image_format_conversion(original_img_path)
        )

        # Run the preprocessing pipeline.
        processed_face, log_status = process_pipeline(
            processed_img_path,
            rej_folder,
        )

        if processed_face is not None:
            output_name = (
                f"{code_prefix}_{str(serial_counter).zfill(4)}.png"
            )

            out_file_path = os.path.join(
                tgt_folder,
                output_name,
            )

            # Save the standardized L0 baseline image as PNG.
            cv2.imwrite(
                out_file_path,
                processed_face,
            )

            serial_counter += 1
            accepted_count += 1

        else:
            rejected_count += 1
            print(
                f"Rejected [{filename}] "
                f"| Reason: {log_status}"
            )

        # Delete temporary files created during HEIC conversion.
        if is_temp_file:
            try:
                os.remove(processed_img_path)

            except Exception as e:
                print(
                    "Error deleting temporary file "
                    f"{processed_img_path}: {e}"
                )

    audit_summary[eth] = {
        "Baseline": accepted_count,
        "Rejected": rejected_count,
    }

    print(
        f"Completed [{eth.upper()}] "
        f"| Accepted: {accepted_count} "
        f"| Rejected: {rejected_count}"
    )

# Aggregate processing summary
print("\n" + "=" * 58)
print("PREPROCESSING SUMMARY")
print("=" * 58)
print(
    "DEMOGRAPHIC GROUP | "
    "L0 BASELINE | "
    "REJECTED"
)
print("-" * 58)

grand_total_good = 0
grand_total_bad = 0

for race in ethnicities:
    good = audit_summary[race]["Baseline"]
    bad = audit_summary[race]["Rejected"]

    grand_total_good += good
    grand_total_bad += bad

    print(
        f"{race.title().ljust(17)} | "
        f"{str(good).ljust(11)} | "
        f"{bad}"
    )

print("-" * 58)
print(
    f"{'Total'.ljust(17)} | "
    f"{str(grand_total_good).ljust(11)} | "
    f"{grand_total_bad}"
)
print("=" * 58)

In [ ]:
ethnicities = ["malay", "chinese", "indian"]
eth_codes = {"malay": "01", "chinese": "02", "indian": "03"}

audit_summary = {}

print("==========================================================")
print("🏁 INITIALIZING ADVANCED DATA SELECTION PIPELINE WORKFLOW ")
print("==========================================================")

for eth in ethnicities:
    src_folder = os.path.join(INPUT_DIR, eth)
    tgt_folder = os.path.join(OUTPUT_DIR, eth)
    rej_folder = os.path.join(REJECTED_DIR, eth)

    os.makedirs(tgt_folder, exist_ok=True)
    os.makedirs(rej_folder, exist_ok=True)

    if not os.path.exists(src_folder):
        print(f"⚠️ Target segment source folder missing: '{src_folder}'... skipping cohort.")
        audit_summary[eth] = {"Baseline": 0, "Rejected": 0}
        continue

    # Scan files with case-insensitive validation support
    all_files = os.listdir(src_folder)
    valid_images = sorted([
        f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.heic'))
    ])

    print(f"\n📂 Processing cohort cluster: [{eth.upper()}] | Found raw total of {len(valid_images)} samples.")

    accepted_count = 0
    rejected_count = 0
    serial_counter = 1
    code_prefix = eth_codes[eth]

    for filename in valid_images:
        original_img_path = os.path.join(src_folder, filename)

        # Handle HEIC conversion before processing
        processed_img_path, is_temp_file = handle_image_format_conversion(original_img_path)

        # Execute the refined multi-stage validation chain
        processed_face, log_status = process_pipeline(processed_img_path, rej_folder)

        if processed_face is not None:
            # Build zero-padded serialized signature mapping index (e.g., 02_0014.png)
            output_name = f"{code_prefix}_{str(serial_counter).zfill(4)}.png"
            out_file_path = os.path.join(tgt_folder, output_name)

            # Commit baseline as a lossless PNG control variable
            cv2.imwrite(out_file_path, processed_face)

            serial_counter += 1
            accepted_count += 1
        else:
            rejected_count += 1
            print(f"  ❌ Omitted [{filename}]: Reason -> {log_status}")

        # Clean up temporary file if one was created during conversion
        if is_temp_file:
            try:
                os.remove(processed_img_path)
            except Exception as e:
                print(f"Error deleting temporary file {processed_img_path}: {e}")

    # Save statistics for output diagnostics
    audit_summary[eth] = {"Baseline": accepted_count, "Rejected": rejected_count}
    print(f"✔️ Cohort completed. Retained: {accepted_count} | Removed: {rejected_count}")

# --- FINAL DIAGNOSTIC AUDIT REPORT SUMMARY TABLE ---
print("\n" + "="*54)
print("             CROSS-DEMOGRAPHIC OPERATIONAL YIELD        ")
print("="*54)
print(" COHORT SEGMENT  |  BASELINE GALLERY  |  REJECT LOG METRIC")
print("-"*54)
grand_total_good = 0
grand_total_bad = 0

for race in ethnicities:
    good = audit_summary[race]["Baseline"]
    bad = audit_summary[race]["Rejected"]
    grand_total_good += good
    grand_total_bad += bad
    print(f" {race.ljust(15)} |  {str(good).ljust(17)} |  {str(bad).ljust(15)}")

print("-"*54)
print(f" {'TOTAL POOL'.ljust(15)} |  {str(grand_total_good).ljust(17)} |  {str(grand_total_bad).ljust(15)}")
print("="*54)

## Output of This Stage

The accepted images produced by this notebook form the standardized **L0 baseline condition** for the compression experiment.

### Output structure

```text
data/
└── preprocessed_images/
    ├── malay/
    ├── chinese/
    └── indian/
```

Each output image:

- contains the isolated facial region produced by the preprocessing pipeline;
- is resized to **112 × 112 pixels**;
- is stored as a PNG file;
- is used as the baseline input for the image-compression stage.

The next notebook, **`02_image_compression.ipynb`**, applies JPEG, JPEG XL, and HEIC compression to these standardized baseline images.